# Feature Engineering:
The process of using domain knowledge to extract features from raw data. These features can be used to improve the performance of machine learning algorithms.

---

## Types:
- Feature Transformation
- Feature Construction
- Feature Selection
- Feature Extraction

---

# Feature Engineering: The Four Categories

Before diving in, here's the umbrella context — feature engineering is generally split into four categories:

| Category | Purpose |
|---|---|
| **Feature Transformation** | Change the *scale/distribution/representation* of existing features (values change, count of features usually stays the same) |
| **Feature Construction/Extraction** | *Create new* features from existing ones (e.g., PCA components, combining columns) |
| **Feature Selection** | *Choose a subset* of existing features, discarding the rest (no new values created) |
| **Feature Extraction** | Reduce dimensionality by deriving new compressed representations (e.g., PCA, LDA, autoencoders) |

---

# Feature Construction

**Category: Feature Construction is one of the four core categories of Feature Engineering itself** — distinct from Feature Transformation, Feature Selection, and Feature Extraction.

## What Is Feature Construction?

Feature Construction is the process of **manually creating new features** from one or more existing features, using domain knowledge, logical reasoning, or mathematical operations — with the goal of exposing patterns or relationships to the model that aren't obvious from the raw columns alone.

The core idea: raw data often doesn't contain the *most useful* representation of information for a model. A human (using domain understanding) can often construct a feature that captures a relationship the model would otherwise have to learn indirectly (or might miss entirely) — for example, a model might struggle to learn that "height and weight together determine BMI," but if you construct the BMI feature yourself, the model gets that signal directly.

---

## Why Feature Construction Matters

1. **Exposes hidden relationships** — some patterns are only visible when features are combined (e.g., ratio, difference, interaction).
2. **Improves model performance** — well-constructed features can significantly boost accuracy, sometimes more than switching to a fancier algorithm.
3. **Injects domain knowledge** — a model has no inherent understanding of your business/domain; constructed features let you encode that expertise directly into the data.
4. **Simplifies what the model needs to learn** — instead of forcing the model to approximate a complex relationship from raw inputs, you hand it a pre-computed, meaningful signal.

---

## Common Techniques

### 1. Combining Multiple Features

Creating a new feature by mathematically combining two or more existing columns.

**a. Ratios**
$$
\text{BMI} = \frac{\text{weight (kg)}}{\text{height (m)}^2}
$$
```python
df['bmi'] = df['weight'] / (df['height'] ** 2)
```

**b. Differences**
$$
\text{Age at Purchase} = \text{Purchase Date} - \text{Birth Date}
$$
```python
df['tenure_days'] = (df['end_date'] - df['start_date']).dt.days
```

**c. Sums/Aggregates**
```python
df['total_spend'] = df['grocery_spend'] + df['electronics_spend'] + df['clothing_spend']
```

**d. Products (Interaction Terms)**
Captures how two features *together* affect the outcome, in a way each feature alone can't:
```python
df['price_x_quantity'] = df['price'] * df['quantity']
```

### 2. Extracting Sub-Components from a Single Feature (a.k.a. "Feature Splitting")
Breaking a single complex feature into multiple simpler, more informative ones.

**a. Date/Time Decomposition**
```python
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['hour'] = df['date'].dt.hour
```
- A raw timestamp is nearly useless to most models directly, but decomposed components (month, day-of-week, is-weekend, hour) often carry strong predictive signal (e.g., retail sales patterns differ by day-of-week and season).

**b. Text Parsing**
```python
df['title'] = df['name'].str.extract(r',\s*([^\.]*)\.')  # e.g., extracting "Mr./Mrs./Dr." from names
df['domain'] = df['email'].str.split('@').str[1]
```

**c. Address/Location Splitting**
```python
df['city'] = df['full_address'].str.split(',').str[0]
df['zip_code'] = df['full_address'].str.extract(r'(\d{5})$')
```

### 3. Binary/Flag Features

Creating a simple 0/1 indicator based on a condition — often used to flag a meaningful threshold or event.

```python
df['has_children'] = (df['num_dependents'] > 0).astype(int)
df['is_high_value_customer'] = (df['total_spend'] > 10000).astype(int)
df['is_missing_income'] = df['income'].isna().astype(int)  # this is the Missing Indicator from earlier!
```

*(Notice — the Missing Indicator technique from your earlier notebook is technically a form of Feature Construction — you're constructing a brand-new binary feature from existing missingness information.)*

### 4. Aggregation-Based Features (especially for grouped/relational data)

Creating features that summarize information across related rows — very common in datasets with a "one-to-many" relationship (e.g., one customer, many transactions).

```python
customer_features = df.groupby('customer_id').agg(
    avg_purchase=('amount', 'mean'),
    total_purchases=('amount', 'count'),
    max_purchase=('amount', 'max')
).reset_index()
```
- Turns transaction-level data into customer-level features — a classic technique in domains like finance, e-commerce, and churn prediction.

### 5. Polynomial Features

Automatically generating higher-order combinations of existing numeric features (squares, cubes, cross-products), useful for capturing non-linear relationships in linear models.

```python
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X[['x1', 'x2']])
# generates: x1, x2, x1^2, x1*x2, x2^2
```
- Useful for linear models that otherwise can't capture non-linear relationships — by explicitly constructing $x_1^2$, $x_1 x_2$, etc. as new features, a linear model can now "see" non-linear patterns.
- **Caution:** the number of generated features grows fast with degree and number of input features — can lead to a large feature space and overfitting risk.

### 6. Domain-Specific Feature Construction

Highly context-dependent — requires actual domain expertise, not a generic formula.

- **Finance:** Debt-to-income ratio, moving averages of stock prices
- **Healthcare:** Body Mass Index (BMI), blood pressure category from raw readings
- **E-commerce:** Recency-Frequency-Monetary (RFM) scores for customer segmentation
- **NLP:** Word count, sentence length, presence of specific keywords

---

## Feature Construction vs. Feature Extraction — Key Distinction

This is a common point of confusion, worth being precise about:

| Aspect | Feature Construction | Feature Extraction |
|---|---|---|
| Driven by | Domain knowledge, manual logic | Mathematical/statistical algorithms |
| Typical effect on feature count | Increases (adds new features alongside/instead of raw ones) | Decreases (compresses many features into fewer) |
| Interpretability | High — you know exactly what "BMI" or "is_weekend" means | Often low — e.g., "Principal Component 1" has no direct real-world meaning |
| Examples | Ratios, date decomposition, domain formulas | PCA, LDA, Autoencoders, t-SNE |
| Human involvement | High — requires understanding of the problem domain | Low — largely automated, data-driven |

**Simple way to remember it:** Construction = *you* build a new feature using logic you understand. Extraction = *an algorithm* compresses existing features into new ones you often can't directly interpret.

---

## Where Feature Construction Fits in the ML Workflow

Since Feature Construction **creates new columns**, it typically happens **before** Feature Transformation and Feature Selection in a real workflow — you construct meaningful new features first, then transform (scale/encode) all features (raw + constructed) together, and finally select which subset actually helps the model.

```
Data Collection
      ↓
Missing Value Handling
      ↓
Feature Construction   ← create new features (ratios, date parts, flags, aggregates)
      ↓
Feature Transformation ← scale/encode all features (raw + newly constructed)
      ↓
Feature Selection      ← keep only the most useful features
      ↓
Model Building
```
---